<a href="https://colab.research.google.com/github/SiyumiJayawardhane/sinhala-wsd/blob/textbook_docai_extraction/Textbook_docai_Extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Textbook → Document AI OCR → target-word sentences (v2)
**PDF → 15-page chunks → Document AI OCR → pages (visual order + confidence) → cleaning → sentence units with page numbers → target-word extraction**. The extraction rules and output columns are the same as the corpus notebook.

### 1. Install

In [2]:
!pip install -q --upgrade google-cloud-documentai pypdf openpyxl

### 2. Settings
* `START = 10`: the first 10 PDF pages are front matter and aren't OCR'd. `book_page = pdf_page - START`.
* `MIN_CONF = 0.85`: pages whose mean **token** confidence is below this are excluded and logged. The threshold comes from SiPaKosa (CHiPSAL 2026).which excluded pages below a mean OCR confidence of 0.85. SiPaKosa used character-level confidence; this notebook uses Document AI’s token confidence.
* `MIN_CHARS, MAX_CHARS, MIN_WORDS, MIN_SI = 20, 400, 4, 0.8`: the sentence filters recorded in the project plan (§4.9).
* outputs stay in `Research-WSD-New/textbooks/Ext_Final/sinhala_lit_g10_11_docai/`.

In [3]:
import os, re, io, json, glob, bisect, random, unicodedata, datetime
from collections import Counter, defaultdict
import pandas as pd
from google.colab import auth, drive
drive.mount("/content/drive")

BASE     = "/content/drive/MyDrive/Research-WSD-New"

PROJECT_ID   = "sinhala-wsd-ocr"
LOCATION     = "us"
PROCESSOR_ID = "7901bb5eb670c044"

BOOK = {
    "book_id":   "sinhala_lit_g10_11",
    "pdf":       f"{BASE}/textbooks/sinhala lit g-10-11.pdf",
    "title":     "සිංහල සාහිත්‍ය සංග්‍රහය (Grade 10–11)",
    "publisher": "Educational Publications Department",
    "grade":     "10-11", "year": "2020", "edition": "7th printing (first printed 2014)",
    "isbn":      "978-955-25-0381-8",
    "url":       "http://edupub.gov.lk/Administrator/Sinhala/10/sin%20lit%20g-10-11/sinhala%20lit%20g-10-11.pdf",
}
# skip front matter and OCR page limit 15
START, CHUNK = 10, 15
MIN_CONF = 0.85
MIN_CHARS, MAX_CHARS, MIN_WORDS, MIN_SI = 20, 400, 4, 0.8

OUT       = f'{BASE}/textbooks/Ext_Final/{BOOK["book_id"]}_docai'
CHUNK_DIR = f"{OUT}/chunks"      # 15-page PDFs made in block 4
RAW       = f"{OUT}/docai_raw"   # one saved OCR response per chunk
for d in (CHUNK_DIR, RAW): os.makedirs(d, exist_ok=True)
print("PDF:", "OK" if os.path.exists(BOOK["pdf"]) else "MISSING", "| out:", OUT)

Mounted at /content/drive
PDF: OK | out: /content/drive/MyDrive/Research-WSD-New/textbooks/Ext_Final/sinhala_lit_g10_11_docai


### 3. Log & Connect

In [4]:
auth.authenticate_user()
from google.api_core.client_options import ClientOptions
from google.cloud import documentai

client = documentai.DocumentProcessorServiceClient(client_options=ClientOptions(
    api_endpoint=f"{LOCATION}-documentai.googleapis.com",
    quota_project_id=PROJECT_ID))

proc = client.get_processor(name=client.processor_path(PROJECT_ID, LOCATION, PROCESSOR_ID))
assert proc.type_ == "OCR_PROCESSOR", f"wrong processor type: {proc.type_}"
assert proc.state.name == "ENABLED", f"processor is {proc.state.name}"

NAME = proc.default_processor_version
VERSION_USED = NAME.split("/")[-1]
ver = client.get_processor_version(name=NAME)
print("processor:", proc.display_name, "| type:", proc.type_, "| version:", VERSION_USED, f"({ver.display_name})")

processor: sinhala-ocr | type: OCR_PROCESSOR | version: pretrained-ocr-v2.1-2024-08-07 (Google Stable)


### 4. Split into 15-page chunks and OCR them
* The PDF is cut into 15-page chunk
* language_hints=["si"] tells the OCR model to expect Sinhala.
* enable_native_pdf_parsing=False: the PDF’s embedded text is legacy FM-Abhaya font (Latin keystrokes), so the OCR must read the page images instead.
* Every request goes to the pinned processor version.
* Responses are saved in docai_raw/

In [5]:
from pypdf import PdfReader, PdfWriter
from tqdm.auto import tqdm

# 1) cut the book into 15-page chunk PDFs (front matter skipped); existing chunks are reused
reader = PdfReader(BOOK["pdf"]); n = len(reader.pages)
chunks = []
for s in range(START, n, CHUNK):
    e = min(s + CHUNK, n); path = f"{CHUNK_DIR}/p{s + 1:03d}-{e:03d}.pdf"
    if not os.path.exists(path):
        w = PdfWriter()
        for i in range(s, e): w.add_page(reader.pages[i])
        w.write(path)
    chunks.append(path)
print("PDF pages:", n, "| chunks:", len(chunks))

# 2) OCR settings: Sinhala hint; read the page images, not the FM-font text layer
OCR_OPTIONS = documentai.ProcessOptions(ocr_config=documentai.OcrConfig(
    hints=documentai.OcrConfig.Hints(language_hints=["si"]), enable_native_pdf_parsing=False))

# 3) OCR each chunk once (pinned version) and save the full response as JSON
errors = []
for path in tqdm(chunks):
    out = f"{RAW}/{os.path.basename(path)[:-4]}.json"
    if os.path.exists(out):
        continue
    try:
        res = client.process_document(request=documentai.ProcessRequest(
            name=NAME, process_options=OCR_OPTIONS,
            raw_document=documentai.RawDocument(content=open(path, "rb").read(), mime_type="application/pdf")))
        expected = len(PdfReader(path).pages)
        assert len(res.document.pages) == expected, f"got {len(res.document.pages)} of {expected} pages"
        with open(out + ".tmp", "w", encoding="utf-8") as f:
            f.write(documentai.Document.to_json(res.document))
        os.replace(out + ".tmp", out)
    except Exception as ex:
        errors.append((os.path.basename(path), repr(ex)[:200]))
print("done:", len(glob.glob(f"{RAW}/p*.json")), "/", len(chunks), "| errors:", errors)

PDF pages: 140 | chunks: 9


  0%|          | 0/9 [00:00<?, ?it/s]

done: 9 / 9 | errors: []


5. Pages in visual order, with confidence

Reads every saved OCR response in docai_raw/ and builds one record per PDF page:

* text: the page rebuilt line by line in visual order. Lines are sorted top to bottom; pieces at the same height (for example a verse line and its rhyme word) are joined left to right.
* ocr_conf: the mean of the page’s token confidences (OCR quality). Used for the MIN_CONF = 0.85 page filter and reported.

In [6]:
def page_rows(page, full_text):
    """Rebuild the page line by line in VISUAL order using each line's position.
       Pieces at the same height (e.g. a verse line + its rhyme word) are joined into one row."""
    W, H = page.dimension.width or 1, page.dimension.height or 1
    items = []
    for ln in page.lines:
        v = ln.layout.bounding_poly.normalized_vertices
        if v:
            xs, ys = [p.x for p in v], [p.y for p in v]
        else:                                                    # only pixel coordinates: convert to 0-1
            v = ln.layout.bounding_poly.vertices
            xs, ys = [p.x / W for p in v], [p.y / H for p in v]
        t = "".join(full_text[int(s.start_index or 0):int(s.end_index)]
                    for s in ln.layout.text_anchor.text_segments).strip()
        if t and xs:
            items.append((min(ys), max(ys), min(xs), t))
    items.sort(key=lambda it: (it[0] + it[1]) / 2)               # top → bottom
    rows = []
    for y0, y1, x, t in items:
        c = (y0 + y1) / 2
        if rows and abs(c - rows[-1]["c"]) < rows[-1]["h"] * 0.5:   # same height → same row
            rows[-1]["parts"].append((x, t))
        else:
            rows.append({"c": c, "h": y1 - y0, "parts": [(x, t)]})
    return "\n".join(" ".join(t for x, t in sorted(r["parts"])) for r in rows)   # left → right

raw_files = sorted(glob.glob(f"{RAW}/p*.json"))
assert raw_files, "no OCR results in docai_raw/ - run block 4 first"
pages = []
for path in raw_files:
    first = int(re.match(r"p(\d+)-", os.path.basename(path)).group(1))     # p011-025.json -> 11
    doc = documentai.Document.from_json(open(path, encoding="utf-8").read(), ignore_unknown_fields=True)
    for k, page in enumerate(doc.pages):
        toks = [t.layout.confidence for t in page.tokens]
        pages.append({"pdf_page": first + k, "printed_page": first + k - START,
                      "ocr_conf": round(sum(toks) / len(toks), 4) if toks else 0.0,
                      "text": page_rows(page, doc.text)})
pages.sort(key=lambda p: p["pdf_page"])

missing = sorted(set(range(START + 1, n + 1)) - {p["pdf_page"] for p in pages})
assert not missing, f"pages missing from the OCR: {missing} - re-run block 4"
with open(f'{OUT}/{BOOK["book_id"]}_pages.jsonl', "w", encoding="utf-8") as f:
    f.writelines(json.dumps(p, ensure_ascii=False) + "\n" for p in pages)
print("pages:", len(pages), "| first:", pages[0]["pdf_page"], "| last:", pages[-1]["pdf_page"])

pages: 130 | first: 11 | last: 140


### 6. Quality check
* Prints the OCR confidence summary and lists the pages below MIN_CONF (0.85). These pages are excluded in block 8.
* Saves sinhala_lit_g10_11_page_confidence.csv (page number + confidence) for the appendix.
* Prints one page so I can compare it with the PDF by eye: reading order, verse lines and rhyme words, and misread letters.

In [13]:
df = pd.DataFrame(pages)
EMPTY = df[df.text.str.strip() == ""].pdf_page.tolist()                 # no words at all (cover, blank, picture)
txt = df[df.text.str.strip() != ""]
LOW = txt[txt.ocr_conf < MIN_CONF].pdf_page.tolist()
print(txt["ocr_conf"].describe().round(4))
print("pages with text:", len(txt), "| mean OCR (token) confidence:", round(txt.ocr_conf.mean(), 4))
print("empty pages:", EMPTY, "| pages with text below", MIN_CONF, ":", LOW)
df[["pdf_page", "printed_page", "ocr_conf"]].to_csv(f'{OUT}/{BOOK["book_id"]}_page_confidence.csv', index=False)

PAGE = 20
sel = df.loc[df.printed_page == PAGE, "text"]
print(f"\n--- printed page {PAGE} (PDF page {PAGE + START}) ---")
print(sel.values[0][:800] if len(sel) else f"no page {PAGE}")

count    129.0000
mean       0.9763
std        0.0055
min        0.9602
25%        0.9735
50%        0.9771
75%        0.9808
max        0.9854
Name: ocr_conf, dtype: float64
pages with text: 129 | mean OCR (token) confidence: 0.9763
empty pages: [140] | pages with text below 0.85 : []

--- printed page 20 (PDF page 30) ---
සිය පින් සිරින් සරු
දෙතිස් ලකුණෙන් විසිතුරු
කෙලෙසුන් කෙරෙන් gór
වඳිම් මුනි උතුමන් තිලෝගුරු
යනුවෙන් ලඝු අකුරු දෙකක් හා ගුරු අකුරකින් යුතු ‘වායු ගණයෙන්’ කාව්‍යය ආරම්භ
කළේ ද එහි අනිෂ්ට විපාක දැන දැන ම බව ජනප්‍රවාදයේ එයි.
1. ගුත්තිල ඇදුරු සඳ
වීණා වයන ලෙස ඉඳ
පසෙක පනවන ලද
සුදුසු අස්නෙක ගොසින් උන් සඳ
අන්වය* : ගුත්තිල ඇදුරු සඳ ඉඳ වීණා වයන ලෙස පසෙක පනවන ලද සුදුසු අනෙක
ගොසින් උන් සඳ,
අර්ථය : ගුත්තිල ඇදුරුතුමා ඉඳ ගෙන වීණා වයනු පිණිස පසෙක පනවන ලද සුදුසු
අස්නෙක ගොසින් උන් කල්හි,
විස්තර (1) පසෙක පනවන ලද : පැනවීම නම් නියම කිරීම යි. කාටත් පොදුවේ නොව
මේ අසවලාට ය, යන නියමය ඇතිව කෙනකුට සුදුසු පරිදි පිළියෙල කරන ලද
අසුන, පනවන ලද අසුන යි.
2. නො දැන ම තමා බල
සබයට අවුත් නො පැකිළ
වැඳ රද පද 

### 7. Cleaning rules
These are your rules, with four fixes:
* **Normalise**: NFC; remove soft hyphen, zero-width space and BOM (U+00AD, U+200B, U+FEFF); keep ZWJ/ZWNJ (part of Sinhala spelling); curly quotes → straight quotes; add a missing space after a full stop.
* **Removed lines**: page numbers, running headers (short lines on 3+ pages), numbered items (“1. “, “3) “), lines under 50% Sinhala, අන්වය / පද බෙදීම blocks, and everything after an exercise heading (අභ්‍යාස / ක්‍රියාකාරකම).
* **Labels kept**: අර්ථය / විස්තර / සටහන. The label is removed and the explanation text is kept.
Verse: a verse-like line is short (<45 characters), has no ending . ? ! : , and is ≥80% Sinhala. Blocks of 2+ such lines are taken out of the prose. A verse is kept as one unit (lines joined with “ / “) when an අන්වය / අර්ථය label follows it.
Sentence splitting: split after . ? !, but not inside initials (ඒ. බී. සී.).
The preview at the end shows, for one page, what is kept, what is removed (with the reason) and the verses found, for comparison with the PDF.


In [21]:
SI = lambda c: "\u0D80" <= c <= "\u0DFF"                     # one Sinhala character
def si_ratio(s):
    chars = [c for c in s if not c.isspace()]
    return sum(SI(c) for c in chars) / len(chars) if chars else 0.0

INVISIBLE  = re.compile(r"[\u00AD\u200B\uFEFF]")            # soft hyphen, zero-width space, BOM
PAGE_NUM   = re.compile(r"^\s*[\divxlcIVXLC]+\s*$")                                   # "20", "ix"
NUMBERED   = re.compile(r"^\s*(\d{1,2}|[ivx]{1,4})\s*[.)]\s")                          # "01. ", "3) "
DROP_LABEL = re.compile(r"^\s*(අන්වය|පද\s*බෙදීම)\s*\*?\s*[:：]")                      # අන්වය block → drop
KEEP_LABEL = re.compile(r"^\s*(අර්ථය|විස්තර\s*(\(\s*\d+\s*\))?|සටහන)\s*\*?\s*[:：]?\s*")  # keep text, drop label
EXERCISE   = re.compile(r"^\s*(අභ්‍යාස|අභ්යාස|ක්‍රියාකාරකම)\s*$")                      # exercise heading
V_LABEL    = re.compile(r"^\s*(අන්වය|අර්ථය|පද\s*බෙදීම)\s*\*?\s*([:：].*)?$")
VERSE_NUM  = re.compile(r"^\s*\d{1,2}\s*\.\s*")

def normalise(s):
    s = INVISIBLE.sub("", unicodedata.normalize("NFC", s))   # ZWJ/ZWNJ kept (Sinhala spelling)
    s = re.sub(r"[“”„]", '"', s)
    s = re.sub(r"[‘’‛]", "'", s)
    s = re.sub(r"\.(?=[\u0D80-\u0DFF])", ". ", s)            # missing space after a full stop
    return re.sub(r"\s+", " ", s).strip()

def is_verse_line(ln):
    return len(ln) < 45 and not re.search(r"[.?!:,]$", ln) and si_ratio(ln) >= 0.8

def repeated_lines(pages):
    """Short lines (<40 chars) that appear on 3 or more pages = running headers / lesson titles."""
    c = Counter()
    for p in pages:
        c.update({ln.strip() for ln in p["text"].split("\n") if 0 < len(ln.strip()) < 40})
    return {ln for ln, k in c.items() if k >= 3}

def clean_page(text, headers):
    """-> (prose text of the page, list of (reason, removed line))"""
    out, removed, mode = [], [], "keep"
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    for i, ln in enumerate(lines):
        if EXERCISE.match(ln):                       # exercises → ignore rest of the page
            removed += [("exercise", l) for l in lines[i:]]; break
        if DROP_LABEL.match(ln):                     # start of අන්වය block
            if not re.search(r"[.?!]\s*$", ln): mode = "skip"
            removed.append(("anvaya", ln)); continue
        if KEEP_LABEL.match(ln):                     # අර්ථය / විස්තර → keep text, remove label
            mode = "keep"; ln = KEEP_LABEL.sub("", ln)
            if not ln: continue
        if mode == "skip":                           # inside අන්වය: skip until a sentence ends
            if re.search(r"[.?!]\s*$", ln): mode = "keep"
            removed.append(("anvaya", ln)); continue
        why = ("page_number" if PAGE_NUM.match(ln) else "header" if ln in headers else
               "numbered_item" if NUMBERED.match(ln) else "non_sinhala" if si_ratio(ln) < 0.5 else "")
        if why: removed.append((why, ln)); continue
        out.append(ln)
    keep, i = [], 0                                  # verse blocks (2+ verse-like lines) leave the prose
    while i < len(out):
        j = i
        while j < len(out) and is_verse_line(out[j]): j += 1
        if j - i >= 2:
            removed += [("verse_block", l) for l in out[i:j]]; i = j
        else:
            keep.append(out[i]); i += 1
    return " ".join(keep), removed

def extract_verses(text):
    """A verse = 2+ verse-like rows directly followed by an අන්වය / අර්ථය label. Rows joined with ' / '."""
    verses, block = [], []
    for ln in (l.strip() for l in text.split("\n")):
        if not ln: continue
        if VERSE_NUM.match(ln):
            block = []; ln = VERSE_NUM.sub("", ln)
            if not ln: continue
        if V_LABEL.match(ln):                        # check the label first: a short label line
            if len(block) >= 2:                      # would otherwise look like a verse line
                verses.append(normalise(" / ".join(block)))
            block = []; continue
        if is_verse_line(ln):
            block.append(ln); continue
        block = []
    return verses

SENT_END = re.compile(r"(?<=[.?!])\s+(?=[\"'\u0D80-\u0DFF])")
INITIALS = {"ඒ","බී","සී","ඩී","ඊ","එෆ්","ජී","එච්","අයි","ජේ","කේ","එල්","එම්","එන්","පී",
            "කිව්","ආර්","එස්","ටී","යූ","ඩබ්ලිව්","ඩබ්ලියු","ඩබ්ලියූ","එක්ස්","වයි","සෙඩ්"}
def ends_with_initial(t):
    w = t.rstrip(".").split(); return bool(w) and w[-1].split(".")[-1] in INITIALS
def starts_with_initial(t):
    w = t.split(); return bool(w) and w[0].endswith(".") and w[0].rstrip(".").split(".")[0] in INITIALS

# --- preview: compare one page with the PDF ---
HEADERS = repeated_lines(pages)
print("running headers found:", sorted(HEADERS))
PAGE = 20                                                    # printed page; PDF page = PAGE + START
p = next(p for p in pages if p["printed_page"] == PAGE)
prose, removed = clean_page(p["text"], HEADERS)
print(f"\n--- printed page {PAGE} (PDF {PAGE + START}) ---\nKEPT PROSE:\n", normalise(prose)[:700])
print("\nREMOVED:"); [print(f"  [{r}] {t}") for r, t in removed]
print("\nVERSES:"); [print("  ", v) for v in extract_verses(p["text"])]

running headers found: ['८८', 'අන්වය : පහසු ය.', 'අභ්\u200dයාස', 'අර්ථය', 'අරුත් පැහැදිලි කිරීම', 'කරන්න.', 'පැහැදිලි කරන්න.', 'පොතේ ඉන්නිසය', 'සාකච්ඡා කරන්න.']

--- printed page 20 (PDF 30) ---
KEPT PROSE:
 යනුවෙන් ලඝු අකුරු දෙකක් හා ගුරු අකුරකින් යුතු 'වායු ගණයෙන්' කාව්‍යය ආරම්භ කළේ ද එහි අනිෂ්ට විපාක දැන දැන ම බව ජනප්‍රවාදයේ එයි. ගුත්තිල ඇදුරුතුමා ඉඳ ගෙන වීණා වයනු පිණිස පසෙක පනවන ලද සුදුසු අස්නෙක ගොසින් උන් කල්හි, පසෙක පනවන ලද : පැනවීම නම් නියම කිරීම යි. කාටත් පොදුවේ නොව මේ අසවලාට ය, යන නියමය ඇතිව කෙනකුට සුදුසු පරිදි පිළියෙල කරන ලද අසුන, පනවන ලද අසුන යි. තමාගේ බලය දන්නේ නැති ව මූසිල ද පැකිළීමක් නැති ව සභාවට අවුත් රජුගේ පා කමල වැඳ, පැත්තක උන් කල්හි, මූසිල පැකිළීමක් නැතිව සභාවට ආවේ තමාගේ බල නො දත් නිසා ය. තම බලයේ පමණ දන්නේ නම් වාදයෙන් තමාට ජය ගත නො හැකි බව වටහා ගනී. එසේ වූ කල සිතේ ධෛර්යයක් ඇති නො වන හෙයින් පැකිළීමක් ඇති වේ. * අන්වය යනු අර්ථය ගැනීම සඳහා පද ගළපන පිළිවෙළ යි.

REMOVED:
  [numbered_item] 1. ගුත්තිල ඇදුරු සඳ
  [anvaya] අන්වය* : ගුත්තිල ඇදුරු සඳ ඉඳ වීණා වයන ලෙස පසෙක පනවන ලද 

[None, None]

8. Build the sentence units
* Every page with text that passes MIN_CONF is cleaned.
* Empty pages are logged as empty_page, and pages below 0.85 as low_confidence.
* Prose: the cleaned pages are joined into one running text, so a sentence that continues onto the next page stays whole. A page map records where each page starts, so every sentence gets the page it starts on.
* Prose is split into sentences (not inside quotes or initials) and cut at numbered notes “(1) …”.
* Verse: every block of 2+ verse-like rows becomes one stanza unit (rows joined with “ / “). It is verse if an අන්වය / අර්ථය label follows it, otherwise verse_nolabel.
* Each unit gets a type: text, verse, verse_nolabel, note (contains “:”), too_short (<20 characters or <4 words), too_long (>400), low_sinhala (<80% Sinhala). Nothing is deleted; every unit and every removed line is saved with its reason.

In [22]:
def all_verse_blocks(text, headers):
    """Every block of 2+ verse-like rows -> (verse joined with ' / ', followed by an අන්වය/අර්ථය label?)"""
    out, block = [], []
    def flush(labelled):
        if len(block) >= 2:
            out.append((normalise(" / ".join(block)), labelled))
        block.clear()
    for ln in (l.strip() for l in text.split("\n")):
        if not ln: continue
        if EXERCISE.match(ln): break                       # same as clean_page: exercises end the page
        if V_LABEL.match(ln):                              # label FIRST: "අන්වය : පහසු ය." is also a repeated line
            flush(True); continue
        if PAGE_NUM.match(ln) or ln in headers:            # page number / header breaks a block
            flush(False); continue
        if VERSE_NUM.match(ln):                            # "2. ..." starts a new verse
            flush(False); ln = VERSE_NUM.sub("", ln)
            if not ln: continue
        if is_verse_line(ln):
            block.append(ln); continue
        flush(False)
    return out

headers = repeated_lines(pages)
removed_log, units = [], []
full, page_starts, page_of = "", [], []
for p in pages:
    if not p["text"].strip():
        removed_log.append({"pdf_page": p["pdf_page"], "reason": "empty_page", "text": ""}); continue
    if p["ocr_conf"] < MIN_CONF:
        removed_log.append({"pdf_page": p["pdf_page"], "reason": "low_confidence", "text": p["text"][:200]}); continue
    prose, removed = clean_page(p["text"], headers)
    removed_log += [{"pdf_page": p["pdf_page"], "reason": r, "text": t} for r, t in removed]
    prose = normalise(prose)
    if prose:
        page_starts.append(len(full)); page_of.append(p); full += prose + " "
full = full.strip()

def page_at(offset):
    return page_of[bisect.bisect_right(page_starts, offset) - 1]

parts = SENT_END.split(full)                         # 1) prose -> sentences
sentences, buf, held = [], "", []
for i, part in enumerate(parts):
    buf = (buf + " " + part).strip() if buf else part
    held.append(part)
    nxt = parts[i + 1] if i + 1 < len(parts) else ""
    if len(buf) <= MAX_CHARS and (ends_with_initial(buf) or starts_with_initial(nxt)):
        continue                                     # part of a name → keep joining
    if buf.count('"') % 2 == 0:                      # quotes balanced → one sentence
        sentences.append(buf); buf, held = "", []
    elif len(buf) > MAX_CHARS:                       # quote never closed (OCR missed it) → keep the sentences separate
        sentences.extend(held); buf, held = "", []
if buf: sentences.append(buf)

def unit_type(s):
    if re.match(r"^[^:]{1,40}\s[:：]\s", s): return "note"
    if len(s) < MIN_CHARS or len(s.split()) < MIN_WORDS: return "too_short"
    if len(s) > MAX_CHARS: return "too_long"
    if si_ratio(s) < MIN_SI: return "low_sinhala"
    return "text"

cursor, sid = 0, 0                                   # 2) prose units with their page
for sent in sentences:
    for s in re.split(r"\(\s*\d+\s*\)", sent):       # cut off numbered notes "(1) …"
        s = s.strip()
        if not s: continue
        pos = full.find(s, cursor)
        pos = pos if pos >= 0 else cursor
        cursor = pos + len(s)
        p = page_at(pos); sid += 1
        units.append({"sid": sid, "pdf_page": p["pdf_page"], "book_page": p["pdf_page"] - START,
                      "type": unit_type(s), "text": s, "ocr_conf": p["ocr_conf"]})

for p in pages:                                      # 3) verses (labelled and unlabelled)
    if not p["text"].strip() or p["ocr_conf"] < MIN_CONF: continue
    for v, labelled in all_verse_blocks(p["text"], headers):
        sid += 1
        typ = ("verse" if labelled else "verse_nolabel") if si_ratio(v) >= MIN_SI else "low_sinhala"
        units.append({"sid": sid, "pdf_page": p["pdf_page"], "book_page": p["pdf_page"] - START,
                      "type": typ, "text": v, "ocr_conf": p["ocr_conf"]})

pd.DataFrame(units).to_csv(f'{OUT}/{BOOK["book_id"]}_units.csv', index=False, encoding="utf-8")
pd.DataFrame(removed_log, columns=["pdf_page", "reason", "text"]).to_csv(
    f'{OUT}/{BOOK["book_id"]}_removed_lines.csv', index=False, encoding="utf-8")

print("units:", dict(Counter(u["type"] for u in units)))
print("removed lines:", dict(Counter(r["reason"] for r in removed_log)))
print("\n--- sample of kept units ---")
show = [u for u in units if u["type"] in ("text", "verse")]
for u in random.Random(1).sample(show, min(6, len(show))):
    print(f'  p{u["book_page"]} [{u["type"]}] {u["text"][:140]}')
print("\n--- verse_nolabel (decide: real poems?) ---")
for u in [u for u in units if u["type"] == "verse_nolabel"][:15]:
    print(f'  p{u["book_page"]} {u["text"][:140]}')
print("\n--- low_sinhala between 50% and 80% (MIN_SI check) ---")
for u in [u for u in units if u["type"] == "low_sinhala" and si_ratio(u["text"]) >= 0.5][:20]:
    print(f'  p{u["book_page"]} {si_ratio(u["text"]):.2f} {u["text"][:140]}')

units: {'text': 1714, 'too_long': 21, 'too_short': 85, 'note': 57, 'low_sinhala': 10, 'verse_nolabel': 84, 'verse': 42}
removed lines: {'verse_block': 905, 'page_number': 120, 'header': 12, 'non_sinhala': 11, 'exercise': 193, 'numbered_item': 60, 'anvaya': 61, 'empty_page': 1}

--- sample of kept units ---
  p23 [text] සින්දේ වීණා තත පමණක් ද?
  p91 [text] කතරගම දෙවි, ස්කන්ධ කුමාර, කාර්තිකේය යන නම්වලින් ද මේ දෙව් රජ හැඳින්වේ.
  p29 [verse] බැඳි දන මන වැයුමෙහි ඇදුරෝයා / විඳි සත්සර වෙණ සර මදුරෝයා / රැඳි සතොසින් සැනහුණු ඉඳුරෝයා / වැඳි දන සිරි ගත්තුය කිඳුරෝයා
  p125 [text] පහ ශ්‍රේණියේ ශිෂ්‍යත්වය සමත්වූ රත්න ශ්‍රී, ද්විතීයික අධ්‍යාපනය සඳහා හික්කඩුවේ ශ්‍රී සුමංගල මධ්‍ය මහා විද්‍යාලයට ඇතුළු විය.
  p117 [text] ඉක්බිතිව ඔවුහු පසුතැවිල්ලෙන් හා වේදනාවෙන් පීඩිතව තම තමන්ගේ මතකයේ සැටියට අර නැති වූ මාලයට සමාන මාලයක් සොයමින් වෙළෙන්දකු වෙළෙන්දකු පාසා යෑමට ප
  p9 [text] මෙසේ රජ්ජුරුවන් මරණ භයින් යුක්ත වැ කියන්නා වූ බස් අසා මහ බෝසතාණන් වහන්සේ, 'මේ රජ්ජුරුවෝ ඉතා මරණ භයින් භයපත් වැ ගියාහ.

--- verse_nolabe

###9. Spot-check file
* 100 random units (text, verse, verse_nolabel; seed 42) are saved to <book>_ocr_spotcheck_100.csv. For each row, open the PDF page (pdf_page) and fill n_word_errors: the number of words the OCR got wrong (a wrong letter, or a missing or extra word; ignore punctuation). n_words = number of Sinhala words in the unit.
* Estimated word error rate (WER) = Σ n_word_errors / Σ n_words, reported together with the mean OCR confidence. The file is never overwritten. If the cleaning rules (blocks 7–8) change, delete it and run again before filling it in.

In [23]:
SI_WORD = re.compile(r"[\u0D80-\u0DFF\u200C\u200D]+")        # Sinhala words only (not " / " or numbers)
path = f'{OUT}/{BOOK["book_id"]}_ocr_spotcheck_100.csv'
if os.path.exists(path):
    print("exists, not overwritten:", path)
else:
    pool = [u for u in units if u["type"] in ("text", "verse", "verse_nolabel")]
    pick = sorted(random.Random(42).sample(pool, min(100, len(pool))), key=lambda u: u["sid"])
    pd.DataFrame([{**u, "n_words": len(SI_WORD.findall(u["text"])), "n_word_errors": "", "notes": ""}
                  for u in pick]).to_csv(path, index=False, encoding="utf-8-sig")
    print("written:", path, "|", len(pick), "units")

exists, not overwritten: /content/drive/MyDrive/Research-WSD-New/textbooks/Ext_Final/sinhala_lit_g10_11_docai/sinhala_lit_g10_11_ocr_spotcheck_100.csv


### 10. Metadata

* Saves <book>_meta.json: the book’s details, the OCR settings (processor version, language hint, chunk size), page confidence, every cleaning and filter setting, and all counts (removed lines by reason, units by type).

In [24]:
meta = {k: v for k, v in BOOK.items() if k != "pdf"}
meta.update({
    "pdf": os.path.basename(BOOK["pdf"]), "page_offset": START, "pages_ocr": f"{START + 1}-{n}",
    "text_method": "Google Document AI Enterprise OCR, online requests in 15-page chunks",
    "processor_type": proc.type_, "processor_version": VERSION_USED, "processor_version_name": ver.display_name,
    "location": LOCATION, "language_hints": ["si"], "native_pdf_parsing": False, "chunk_pages": CHUNK,
    "n_chunks": len(raw_files), "run_date": datetime.date.today().isoformat(),
    "n_pages": len(pages), "empty_pages": EMPTY, "pages_with_text": len(txt),
    "mean_token_conf_pages_with_text": round(txt.ocr_conf.mean(), 4),
    "min_token_conf_pages_with_text": round(txt.ocr_conf.min(), 4),
    "conf_min": MIN_CONF, "conf_min_source": "SiPaKosa (Gurusinghe & Jayatilleke), Sec. 3.2; token instead of character confidence",
    "low_conf_pages": LOW,
    "normalisation": "NFC; remove U+00AD U+200B U+FEFF; keep ZWJ/ZWNJ; curly quotes -> straight; space after full stop",
    "cleaning": {"header": "line <40 chars on >=3 pages", "line_min_sinhala": 0.5,
                 "verse_line": "<45 chars, no ending . ? ! : , and >=80% Sinhala",
                 "verse_unit": "2+ verse rows joined with ' / '; 'verse' if followed by අන්වය/අර්ථය, else 'verse_nolabel'",
                 "dropped_blocks": "අන්වය / පද බෙදීම; everything after an exercise heading"},
    "sentence_filters": {"min_chars": MIN_CHARS, "max_chars": MAX_CHARS, "min_words": MIN_WORDS, "min_sinhala": MIN_SI},
    "removed_lines": dict(Counter(r["reason"] for r in removed_log)),
    "units": dict(Counter(u["type"] for u in units)),
})
json.dump(meta, open(f'{OUT}/{BOOK["book_id"]}_meta.json', "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print(json.dumps({k: meta[k] for k in ("processor_version", "pages_with_text", "mean_token_conf_pages_with_text",
                                        "empty_pages", "low_conf_pages", "units")}, ensure_ascii=False))

{"processor_version": "pretrained-ocr-v2.1-2024-08-07", "pages_with_text": 129, "mean_token_conf_pages_with_text": 0.9763, "empty_pages": [140], "low_conf_pages": [], "units": {"text": 1714, "too_long": 21, "too_short": 85, "note": 57, "low_sinhala": 10, "verse_nolabel": 84, "verse": 42}}


###11. Target-word extraction
* Word list: WSD_Polysemy_Words.xlsx.
* Filters (NOISE, GRAMMATICAL), target_info, reject_reason and dedup_key are copied from the corpus notebook. They must stay identical.
* Uses text and verse units with ≥ 4 Sinhala tokens; exact duplicates per word are skipped; all hits are kept (no sampling).
* Matching: whole token, NFC, ZWJ/ZWNJ ignored. Target = the first occurrence not inside a filter pattern (target_start, target_end; sentence[target_start:target_end] == matched).
* Outputs: <book>_target_word_sentences.csv (every row; filter_reason empty = kept), .txt (kept rows only), _target_word_summary.xlsx. Counts are added to the meta JSON.

In [25]:
import openpyxl

WORDS_XLSX  = f"{BASE}/WSD_Polysemy_Words.xlsx"
WORDS_SHEET = "target_words"
EXPECTED_N  = 70
MIN_TOKENS  = 4
EXTRACT_TYPES = {"text", "verse"}

JOINERS = re.compile(r"[\u200C\u200D]")
TOKEN_PATTERN = re.compile(r"[\u0D80-\u0DFF\u200C\u200D]+")
def match_key(tok):
    """Key used ONLY for matching: NFC + ZWJ/ZWNJ removed, so ප්\u200Dර and ප්ර match the same."""
    return JOINERS.sub("", unicodedata.normalize("NFC", str(tok)))

def load_target_words(path=WORDS_XLSX, sheet=WORDS_SHEET):
    """-> words (sheet order), {word: No}, {word: [spellings]}."""
    rows = openpyxl.load_workbook(path, read_only=True)[sheet].iter_rows(values_only=True)
    header = [str(h).strip() if h else "" for h in next(rows)]
    i_no, i_word = header.index("No"), header.index("Word")
    words, numbers, spellings, seen_no = [], {}, {}, set()
    for r in rows:
        r = tuple(r) + (None,) * len(header)
        if not r[i_word]:
            continue
        w, no = str(r[i_word]).strip(), int(float(r[i_no]))
        if w in numbers or no in seen_no:
            raise ValueError(f"Duplicate word or number in sheet: {no} {w}")
        words.append(w); numbers[w] = no; seen_no.add(no)
        spellings[w] = [s.strip() for s in re.split(r"[/,]", w) if s.strip()]
    return words, numbers, spellings

TARGETS, word_no, word_spellings = load_target_words()
assert len(TARGETS) == EXPECTED_N, f"Expected {EXPECTED_N} words, got {len(TARGETS)}"
MATCH = {}
for w in TARGETS:
    for s in word_spellings[w]:
        k = match_key(s)
        if k in MATCH and MATCH[k][0] != w:
            raise ValueError(f"Spelling {s} belongs to two words: {MATCH[k][0]} and {w}")
        MATCH[k] = (w, s)

# ---- filters: COPY of the corpus notebook cell 7 (keep the two identical) ----
S = r"[\u0D80-\u0DFF\u200C\u200D]"
NOISE = {
    "රැස": [r"ඇස\s*පාදන\s*රැස"],                                         # website column title
    "සල්": [r"\(\s*සල්\s*\)"],                                            # Islamic honorific abbreviation
    "සී":  [rf"(?<!{S})සී\s*\.", r"විටමින්\s*සී"],                       # English letter C (initials, vitamin C)
    "මී":  [r"මී\s*ළඟ", r"මී\s*ළග"],                                     # split මීළඟ ('next')
    "දන":  [r"දන\s*(?:ගහ|ගැහ|ගැස|ගසා|ගස්ස)"],                         # misspelt දණ ගහනවා ('kneel')
}
#selected as senses
GRAMMATICAL = {
    #"කර":  [rf"කර\s*(?:ඇත|ඇති|තිබේ|තිබුණි|තිබෙන|තිබූ|ගැනීම|ගන්න|ගත්|ගෙන|දී|සිටි|සිටින|නැත)(?!{S})"],   'කර ඇත' = has done
    #"කලි": [r"වූ\s*කලි"],                                                  'වූ කලි' = when
    #"වස්": [r"නු\s*වස්"],                                                  'කරනු වස්' = in order to
}

def _compile(table):
    out = {}
    for spelling, pats in table.items():
        k = match_key(spelling)
        assert k in MATCH, f"Filter key '{spelling}' is not a spelling in the word list"
        out.setdefault(MATCH[k][0], []).extend(re.compile(p) for p in pats)
    return out
FILTERS = {"noise": _compile(NOISE), "grammatical": _compile(GRAMMATICAL)}

def reject_reason(word, sent, n_hits):
    """'noise' / 'grammatical' if every occurrence of the word is inside such a pattern, else None."""
    for reason, table in FILTERS.items():
        pats = table.get(word)
        if pats and sum(len(p.findall(sent)) for p in pats) >= n_hits:
            return reason
    return None

def target_info(sentence, word):
    """-> (target_start, target_end, n_occurrences): first occurrence not inside a filter pattern."""
    spans = [(m.start(), m.end()) for m in TOKEN_PATTERN.finditer(sentence)
             if MATCH.get(match_key(m.group()), (None,))[0] == word]
    blocked = set()
    for table in FILTERS.values():
        for p in table.get(word, []):
            for m in p.finditer(sentence):
                blocked |= {i for i, (s, e) in enumerate(spans) if s >= m.start() and e <= m.end()}
    free = [sp for i, sp in enumerate(spans) if i not in blocked]
    s, e = (free or spans)[0]
    return s, e, len(spans)

def dedup_key(s):
    return re.sub(r"[^\u0D80-\u0DFF0-9]", "", match_key(s))

# ---- scan the units ----
stats, seen, rows = Counter(), defaultdict(set), []
for u in units:
    if u["type"] not in EXTRACT_TYPES:
        stats[f'skipped_{u["type"]}'] += 1; continue
    sent = u["text"]
    if len(TOKEN_PATTERN.findall(sent)) < MIN_TOKENS:
        stats["skipped_short"] += 1; continue
    hits = Counter()
    for tok in TOKEN_PATTERN.findall(sent):
        m = MATCH.get(match_key(tok))
        if m: hits[m[0]] += 1
    h = dedup_key(sent)
    for word, n_hits in hits.items():
        if h in seen[word]:
            stats["skipped_duplicate"] += 1; continue
        seen[word].add(h)
        ts, te, n_occ = target_info(sent, word)
        rows.append({"No": word_no[word], "word": word, "matched": sent[ts:te],
                     "source": "textbook", "sub_source": BOOK["book_id"], "doc_id": u["pdf_page"],
                     "sentence": sent, "target_start": ts, "target_end": te, "n_occurrences": n_occ,
                     "filter_reason": reject_reason(word, sent, n_hits) or "",
                     "unit_type": u["type"], "sid": u["sid"], "pdf_page": u["pdf_page"],
                     "book_page": u["book_page"], "ocr_conf": u["ocr_conf"]})

COLS = ["No", "word", "matched", "source", "sub_source", "doc_id", "sentence",          # same as corpus pool
        "target_start", "target_end", "n_occurrences",
        "filter_reason", "unit_type", "sid", "pdf_page", "book_page", "ocr_conf"]       # textbook extras
cand = pd.DataFrame(rows, columns=COLS).sort_values(["No", "sid"])
assert all(r.sentence[r.target_start:r.target_end] == r.matched for r in cand.itertuples())
assert all(MATCH[match_key(r.matched)][0] == r.word for r in cand.itertuples())
cand.to_csv(f'{OUT}/{BOOK["book_id"]}_target_word_sentences.csv', index=False, encoding="utf-8")

kept = cand[cand.filter_reason == ""]
with open(f'{OUT}/{BOOK["book_id"]}_target_word_sentences.txt', "w", encoding="utf-8") as f:
    f.write("\n\n".join(f"s_word: {w}\nsentences: " + "; ".join(s.replace(";", ",") for s in kept.loc[kept.word == w, "sentence"])
                        for w in TARGETS))
summary = pd.DataFrame([{
    "No": word_no[w], "word": w, "matches_kept": int((kept.word == w).sum()),
    "rejected_noise": int(((cand.word == w) & (cand.filter_reason == "noise")).sum()),
    "rejected_grammatical": int(((cand.word == w) & (cand.filter_reason == "grammatical")).sum()),
    "by_spelling": ", ".join(f"{k} ({v})" for k, v in kept.loc[kept.word == w, "matched"].value_counts().items()),
    "by_unit_type": ", ".join(f"{k} ({v})" for k, v in kept.loc[kept.word == w, "unit_type"].value_counts().items()),
} for w in TARGETS])
summary.to_excel(f'{OUT}/{BOOK["book_id"]}_target_word_summary.xlsx', index=False)

meta_path = f'{OUT}/{BOOK["book_id"]}_meta.json'; meta = json.load(open(meta_path, encoding="utf-8"))
meta.update({"word_list": os.path.basename(WORDS_XLSX), "word_sheet": WORDS_SHEET, "n_words": len(TARGETS),
             "matching": "base form only; any listed spelling = same word; whole tokens; NFC; ZWJ/ZWNJ-insensitive",
             "min_tokens": MIN_TOKENS, "extract_types": sorted(EXTRACT_TYPES),
             "dedup": "exact, per word (Sinhala letters + digits)",
             "noise_patterns": NOISE, "grammatical_patterns": GRAMMATICAL,
             "extraction_counts": dict(stats),
             "filter_counts": cand.filter_reason.replace("", "kept").value_counts().to_dict(),
             "target_position_rule": "first occurrence not inside a filter pattern",
             "matches_kept_total": len(kept),
             "words_with_0": summary.loc[summary.matches_kept == 0, "word"].tolist()})
json.dump(meta, open(meta_path, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print(f"{len(cand)} rows | kept {len(kept)} | {dict(stats)}")
print("filtered:", cand.filter_reason.replace("", "kept").value_counts().to_dict())
print("words with 0:", meta["words_with_0"])

589 rows | kept 589 | {'skipped_too_long': 21, 'skipped_too_short': 85, 'skipped_note': 57, 'skipped_low_sinhala': 10, 'skipped_verse_nolabel': 84}
filtered: {'kept': 589}
words with 0: ['දද', 'රැස', 'කලාපය', 'තිර', 'හස / හස්', 'පණ', 'කලි', 'පට', 'සමර', 'කුළු', 'දීප', 'වණ', 'දල', 'නිදන', 'මාන', 'තිත්']
